# 05_XAI_PC_evaluation_alignment_prob

## Chance-adjusted pseudo-concept evaluation

This notebook supplements the existing pseudo-concept evaluation notebook.

The aim is **not** to replace Dice, IoU, SIR, or TIxAI/TAxAI.

Instead, this notebook asks:

> Does the XAI map align with the pseudo-concept more than would be expected by chance inside the lesion?

This is useful because pseudo-concept regions such as asymmetry and border irregularity are often small, sparse, thin, or disconnected. Raw Dice/IoU values can therefore be low even when the overlap is better than a random lesion-region baseline.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

## 1. Configuration

These paths and constants are reused from the unified Phase 5 evaluation notebook.

If this notebook is run from `src/`, keep `ROOT = Path("..").resolve()`.

If it is run from the repository root, change it to `ROOT = Path(".").resolve()`.

In [ ]:
# If your notebook is in src/, ROOT = Path("..").resolve() is usually correct.
# If you run this notebook from the repository root, change ROOT = Path(".").resolve().

ROOT = Path("..").resolve()

DATA_DIR = ROOT / "data"
OUTPUTS_DIR = ROOT / "outputs"
RUN_NAME = "full_dataset" # change run_name to match the run you want to analyze

RUN_CSV = DATA_DIR / f"{RUN_NAME}" / f"{RUN_NAME}.csv"

XAI_MANIFEST = OUTPUTS_DIR / f"{RUN_NAME}" / "manifests" /  f"{RUN_NAME}_xai_manifest.csv"
PSEUDO_MANIFEST = OUTPUTS_DIR / f"{RUN_NAME}" / "manifests" / f"{RUN_NAME}_pseudo_concepts_manifest.csv"

PHASE5_DIR = OUTPUTS_DIR / f"phase5_{RUN_NAME}"
METRICS_DIR = PHASE5_DIR / "metrics"
FIG_DIR = PHASE5_DIR / "figures"

CHANCE_METRICS_DIR = METRICS_DIR / "chance_adjusted"
CHANCE_FIG_DIR = FIG_DIR / "chance_adjusted"

CHANCE_METRICS_DIR.mkdir(parents=True, exist_ok=True)
CHANCE_FIG_DIR.mkdir(parents=True, exist_ok=True)

XAI_METHODS = ["gradcam", "lime", "shap"]

# These keys must match the keys saved inside the pseudo-concept .npz files.
CONCEPT_KEYS = {
    "asymmetry": "asymmetry",
    "border_default": "border_irregularity",
    "border_w8_dil2_sigma5": "border_w8_dil2_sigma5",
    "border_w12_dil4_sigma8": "border_w12_dil4_sigma8",
    "border_w16_dil6_sigma10": "border_w16_dil6_sigma10",
    "border_w16_dil6_sigma10_dist": "border_w16_dil6_sigma10_dist",
    "colour_heterogeneity": "colour_heterogeneity",
}

TOP_K_PERCENT = 20
N_RANDOM_SAMPLES = 100
SAVE_RANDOM_DISTRIBUTIONS = True
CHECKPOINT_EVERY = 250
RANDOM_STATE = 42
EPS = 1e-8

# If True, pseudo-concept pixels are restricted to the lesion mask before comparison.
# This is recommended because the random baseline is also sampled inside the lesion.
RESTRICT_CONCEPT_TO_LESION = True

print("ROOT:", ROOT)
print("Pilot CSV:", RUN_CSV, "exists:", RUN_CSV.exists())
print("XAI manifest:", XAI_MANIFEST, "exists:", XAI_MANIFEST.exists())
print("Pseudo manifest:", PSEUDO_MANIFEST, "exists:", PSEUDO_MANIFEST.exists())
print("Chance metrics dir:", CHANCE_METRICS_DIR)
print("Chance figures dir:", CHANCE_FIG_DIR)

## 2. Load and merge manifests

This mirrors the unified evaluation notebook:

- run subset CSV
- XAI manifest
- pseudo-concept manifest

The merged dataframe is called `run_resolved`.

In [ ]:
def normalise_dataset_name(x):
    x = str(x).strip().lower()
    aliases = {
        "ham": "ham10000",
        "ham10000": "ham10000",
        "isic": "isic2018",
        "isic2018": "isic2018",
    }
    return aliases.get(x, x)


def ensure_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


ensure_exists(RUN_CSV, "Pilot CSV")
ensure_exists(XAI_MANIFEST, "XAI manifest")
ensure_exists(PSEUDO_MANIFEST, "Pseudo-concept manifest")

run_csv = pd.read_csv(RUN_CSV)
xai_manifest = pd.read_csv(XAI_MANIFEST)
pseudo_manifest = pd.read_csv(PSEUDO_MANIFEST)

for name, df in [
    ("run_csv", run_csv),
    ("xai_manifest", xai_manifest),
    ("pseudo_manifest", pseudo_manifest),
]:
    if "dataset" not in df.columns or "stem" not in df.columns:
        raise ValueError(f"{name} must contain columns: dataset, stem")

    df["dataset"] = df["dataset"].map(normalise_dataset_name)
    df["stem"] = df["stem"].astype(str)

run_resolved = (
    run_csv
    .merge(
        xai_manifest,
        on=["dataset", "stem"],
        how="left",
        suffixes=("", "_xai"),
    )
    .merge(
        pseudo_manifest,
        on=["dataset", "stem"],
        how="left",
        suffixes=("", "_pseudo"),
    )
)

print("Run rows:", len(run_csv))
print("XAI manifest rows:", len(xai_manifest))
print("Pseudo manifest rows:", len(pseudo_manifest))
print("Merged run rows:", len(run_resolved))

for method in XAI_METHODS:
    col = f"xai_{method}_path"
    if col in run_resolved.columns:
        print(f"Rows with {col}:", run_resolved[col].notna().sum())
    else:
        print(f"Missing column: {col}")

if "pseudo_npz_path" in run_resolved.columns:
    print("Rows with pseudo_npz_path:", run_resolved["pseudo_npz_path"].notna().sum())
else:
    print("Missing column: pseudo_npz_path")

display(run_resolved.head())

## 3. Path validation

This checks whether the XAI map paths and pseudo-concept NPZ paths resolve correctly.

In [ ]:
def resolve_path(p):
    if p is None or pd.isna(p):
        return None

    p = Path(str(p))

    if p.is_absolute():
        return p

    # First try relative to the current notebook working directory.
    if p.exists():
        return p.resolve()

    # Then try relative to repository root.
    p2 = ROOT / p
    if p2.exists():
        return p2.resolve()

    return p


path_check_records = []

for _, row in run_resolved.iterrows():
    rec = {
        "dataset": row["dataset"],
        "stem": row["stem"],
    }

    for method in XAI_METHODS:
        col = f"xai_{method}_path"
        p = resolve_path(row.get(col)) if col in run_resolved.columns else None
        rec[f"{method}_exists"] = bool(p is not None and Path(p).exists())

    p = resolve_path(row.get("pseudo_npz_path")) if "pseudo_npz_path" in run_resolved.columns else None
    rec["pseudo_exists"] = bool(p is not None and Path(p).exists())

    path_check_records.append(rec)

path_check = pd.DataFrame(path_check_records)

print("File availability summary:")
display(path_check.drop(columns=["dataset", "stem"]).sum().to_frame("count"))

missing_any = path_check[
    ~path_check[[f"{m}_exists" for m in XAI_METHODS] + ["pseudo_exists"]].all(axis=1)
]

print("Rows missing at least one required file:", len(missing_any))
display(missing_any.head(20))

## 4. Map loading and metric helpers

These functions are reused from the unified evaluation notebook.

Important detail:

- Dice and IoU use a binary top-k XAI map.
- SIR and TIxAI use the continuous saliency map.
- The random baseline samples a pseudo-concept-sized binary region inside the lesion mask.

In [ ]:
# Min-max normalize to [0,1], handle common channel-first/last cases, and ensure 2D output.
def normalize_map(x):
    arr = np.asarray(x)
    arr = np.squeeze(arr)

    # Convert common channel-first or channel-last attribution arrays to 2D.
    if arr.ndim == 3:
        if arr.shape[0] in [1, 3, 4]:
            arr = np.mean(np.abs(arr), axis=0)
        elif arr.shape[-1] in [1, 3, 4]:
            arr = np.mean(np.abs(arr), axis=-1)

    if arr.ndim != 2:
        raise ValueError(f"Expected 2D map after conversion, got shape {arr.shape}")

    arr = arr.astype(np.float32)

    if not np.isfinite(arr).all():
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    arr = arr - arr.min()
    denom = arr.max() + EPS
    arr = arr / denom

    return arr


def load_map_file(path, key=None):
    path = resolve_path(path)
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()

    if suffix == ".npy":
        return normalize_map(np.load(path, allow_pickle=False))

    if suffix == ".npz":
        data = np.load(path, allow_pickle=False)

        if key is not None:
            if key not in data.files:
                raise KeyError(
                    f"Key '{key}' not found in {path}. Available keys: {data.files}"
                )
            return normalize_map(data[key])

        return normalize_map(data[data.files[0]])

    img = Image.open(path).convert("L")
    return normalize_map(np.asarray(img))


def topk_binary(x, top_k_percent=20):
    x = normalize_map(x)
    threshold = np.percentile(x, 100 - top_k_percent)
    return x >= threshold


def binary_from_map(x, threshold=0.5):
    x = normalize_map(x)
    return x >= threshold


def iou_score(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / (union + EPS))


def dice_score(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    inter = np.logical_and(a, b).sum()
    return float((2 * inter) / (a.sum() + b.sum() + EPS))


def saliency_inside_ratio(saliency, region):
    saliency = normalize_map(saliency)
    region = np.asarray(region).astype(bool)

    total = saliency.sum()
    if total <= EPS:
        return 0.0

    return float(saliency[region].sum() / (total + EPS))


def mean_inside_outside_ratio(saliency, region):
    saliency = normalize_map(saliency)
    region = np.asarray(region).astype(bool)

    if region.sum() == 0:
        return 0.0

    inside = saliency[region].mean()

    if (~region).sum() == 0:
        return np.nan

    outside = saliency[~region].mean()

    return float(inside / (outside + EPS))


def pearson_corr(a, b):
    a = normalize_map(a).ravel()
    b = normalize_map(b).ravel()

    if a.std() <= EPS or b.std() <= EPS:
        return 0.0

    return float(np.corrcoef(a, b)[0, 1])


def tixai_score(
    saliency: np.ndarray,
    region: np.ndarray,
    eps: float = EPS,
) -> float:
    '''
    Compute TIxAI-style region concentration score.

    This uses the same implementation as the unified evaluation notebook:
    mean saliency inside the region divided by inside + outside mean saliency.
    '''
    saliency = normalize_map(saliency).astype(np.float32)
    mask = region.astype(bool)

    inside = saliency[mask]
    outside = saliency[~mask]

    lesion_relevance = inside.mean() if inside.size > 0 else 0.0
    background_relevance = outside.mean() if outside.size > 0 else 0.0

    tixai = lesion_relevance / (
        lesion_relevance + background_relevance + eps
    )

    return float(tixai)


def safe_divide(numerator, denominator, eps=EPS):
    if denominator is None or not np.isfinite(denominator) or abs(denominator) <= eps:
        return np.nan
    return float(numerator / denominator)


def safe_z_score(observed, random_mean, random_std, eps=EPS):
    if random_std is None or not np.isfinite(random_std) or random_std <= eps:
        return np.nan
    return float((observed - random_mean) / random_std)

## 5. Random same-area lesion-region baseline

For each image, XAI method, and pseudo-concept:

1. Compute the observed XAI to pseudo-concept overlap.
2. Count the number of pseudo-concept pixels inside the lesion.
3. Randomly sample the same number of pixels from inside the lesion mask.
4. Compute the same metrics against this random region.
5. Repeat this `N_RANDOM_SAMPLES` times.
6. Compare the observed score against the random distribution.

The random region deliberately does **not** preserve shape or continuity. This is appropriate here because the pseudo-concept maps themselves can be sparse, disconnected, and irregular.

In [ ]:
def sample_random_same_area_region(
    lesion_bin: np.ndarray,
    n_pixels: int,
    rng: np.random.Generator,
) -> np.ndarray:
    '''
    Sample n_pixels randomly inside the lesion mask.

    The sampled region is not required to be connected.
    '''
    lesion_bin = np.asarray(lesion_bin).astype(bool)
    region = np.zeros_like(lesion_bin, dtype=bool)

    lesion_indices = np.flatnonzero(lesion_bin.ravel())

    if n_pixels <= 0 or lesion_indices.size == 0:
        return region

    n_pixels = int(min(n_pixels, lesion_indices.size))

    selected = rng.choice(
        lesion_indices,
        size=n_pixels,
        replace=False,
    )

    region.ravel()[selected] = True
    return region


def compute_region_metrics(
    xai_map: np.ndarray,
    xai_bin: np.ndarray,
    region_bin: np.ndarray,
) -> dict:
    '''
    Compute the metrics used for XAI-vs-region comparison.
    '''
    return {
        "iou": iou_score(xai_bin, region_bin),
        "dice": dice_score(xai_bin, region_bin),
        "sir": saliency_inside_ratio(xai_map, region_bin),
        "tixai": tixai_score(xai_map, region_bin),
        "inside_outside_ratio": mean_inside_outside_ratio(xai_map, region_bin),
    }


def empirical_p_greater_or_equal(observed, random_values):
    '''
    One-sided empirical p-value:
    probability that random baseline is at least as high as the observed score.

    Small values suggest observed alignment is higher than expected by chance.
    '''
    random_values = np.asarray(random_values, dtype=float)
    random_values = random_values[np.isfinite(random_values)]

    if random_values.size == 0 or not np.isfinite(observed):
        return np.nan

    return float((np.sum(random_values >= observed) + 1) / (random_values.size + 1))

## 6. Compute chance-adjusted pseudo-concept metrics

Output level: one row per image, XAI method, and pseudo-concept.

For each metric, the notebook saves:

- observed value
- random mean
- random standard deviation
- enrichment = observed / random mean
- z-score = observed minus random mean, divided by random standard deviation
- empirical p-value, one-sided, testing whether the observed value is higher than random

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

import gc

checkpoint_path = CHANCE_METRICS_DIR / f"{RUN_NAME}_chance_adjusted_checkpoint.csv"
errors_checkpoint_path = CHANCE_METRICS_DIR / f"{RUN_NAME}_chance_load_errors_checkpoint.csv"

chance_records = []
random_distribution_records = []
load_errors = []

metric_names = [
    "iou",
    "dice",
    "sir",
    "tixai",
    "inside_outside_ratio",
]

for image_idx, row in tqdm(run_resolved.iterrows(), total=len(run_resolved)):
    dataset = row["dataset"]
    stem = str(row["stem"])

    pseudo_path = row.get("pseudo_npz_path")
    if pseudo_path is None or pd.isna(pseudo_path):
        continue

    pseudo_path = resolve_path(pseudo_path)

    try:
        with np.load(pseudo_path, allow_pickle=False) as data:
            if "mask" not in data.files:
                raise KeyError(f"'mask' key not found. Available keys: {data.files}")
            lesion_map = normalize_map(data["mask"])
            lesion_bin = binary_from_map(lesion_map, threshold=0.5)
    except Exception as e:
        load_errors.append({
            "dataset": dataset,
            "stem": stem,
            "type": "lesion_mask",
            "path": str(pseudo_path),
            "error": str(e),
        })
        continue

    lesion_area_pixels = int(lesion_bin.sum())

    if lesion_area_pixels == 0:
        load_errors.append({
            "dataset": dataset,
            "stem": stem,
            "type": "lesion_mask",
            "path": str(pseudo_path),
            "error": "Empty lesion mask.",
        })
        continue

    for method in XAI_METHODS:
        xai_col = f"xai_{method}_path"

        if xai_col not in run_resolved.columns:
            continue

        xai_path = row.get(xai_col)
        if xai_path is None or pd.isna(xai_path):
            continue

        xai_path = resolve_path(xai_path)

        try:
            xai_map = load_map_file(xai_path)
            xai_bin = topk_binary(xai_map, TOP_K_PERCENT)
        except Exception as e:
            load_errors.append({
                "dataset": dataset,
                "stem": stem,
                "type": "xai",
                "method": method,
                "path": str(xai_path),
                "error": str(e),
            })
            continue

        for concept_name, concept_key in CONCEPT_KEYS.items():
            try:
                concept_map = load_map_file(pseudo_path, key=concept_key)
                concept_bin_raw = binary_from_map(concept_map, threshold=0.5)

                if RESTRICT_CONCEPT_TO_LESION:
                    concept_bin = np.logical_and(concept_bin_raw, lesion_bin)
                else:
                    concept_bin = concept_bin_raw

            except Exception as e:
                load_errors.append({
                    "dataset": dataset,
                    "stem": stem,
                    "type": "concept",
                    "concept": concept_name,
                    "key": concept_key,
                    "path": str(pseudo_path),
                    "error": str(e),
                })
                continue

            concept_area_pixels = int(concept_bin.sum())

            if concept_area_pixels == 0:
                # Keep an explicit row so that empty concepts are visible in the outputs.
                base = {
                    "dataset": dataset,
                    "stem": stem,
                    "xai_method": method,
                    "concept": concept_name,
                    "concept_key": concept_key,
                    "top_k_percent": TOP_K_PERCENT,
                    "n_random_samples": N_RANDOM_SAMPLES,
                    "restricted_concept_to_lesion": RESTRICT_CONCEPT_TO_LESION,
                    "lesion_area_pixels": lesion_area_pixels,
                    "concept_area_pixels": 0,
                    "concept_area_ratio_image": 0.0,
                    "concept_area_ratio_lesion": 0.0,
                    "xai_area_ratio_image": float(xai_bin.mean()),
                    "xai_path": str(xai_path),
                    "pseudo_npz_path": str(pseudo_path),
                }

                for metric in metric_names:
                    base[f"observed_{metric}"] = np.nan
                    base[f"random_{metric}_mean"] = np.nan
                    base[f"random_{metric}_std"] = np.nan
                    base[f"{metric}_enrichment"] = np.nan
                    base[f"{metric}_z"] = np.nan
                    base[f"{metric}_empirical_p_ge"] = np.nan

                base["pearson_corr"] = pearson_corr(xai_map, concept_map)
                chance_records.append(base)
                continue

            observed = compute_region_metrics(xai_map, xai_bin, concept_bin)

            random_metrics = {metric: [] for metric in metric_names}

            for sample_idx in range(N_RANDOM_SAMPLES):
                random_region = sample_random_same_area_region(
                    lesion_bin=lesion_bin,
                    n_pixels=concept_area_pixels,
                    rng=rng,
                )

                sample_metrics = compute_region_metrics(
                    xai_map=xai_map,
                    xai_bin=xai_bin,
                    region_bin=random_region,
                )

                for metric in metric_names:
                    random_metrics[metric].append(sample_metrics[metric])

                # Optional long-form random distributions, useful for diagnostics.
                if SAVE_RANDOM_DISTRIBUTIONS:
                    random_distribution_records.append({
                        "dataset": dataset,
                        "stem": stem,
                        "xai_method": method,
                        "concept": concept_name,
                        "random_sample_idx": sample_idx,
                        **{f"random_{metric}": sample_metrics[metric] for metric in metric_names},
                    })

            base = {
                "dataset": dataset,
                "stem": stem,
                "xai_method": method,
                "concept": concept_name,
                "concept_key": concept_key,
                "top_k_percent": TOP_K_PERCENT,
                "n_random_samples": N_RANDOM_SAMPLES,
                "restricted_concept_to_lesion": RESTRICT_CONCEPT_TO_LESION,
                "lesion_area_pixels": lesion_area_pixels,
                "concept_area_pixels": concept_area_pixels,
                "concept_area_ratio_image": float(concept_bin.mean()),
                "concept_area_ratio_lesion": float(concept_area_pixels / (lesion_area_pixels + EPS)),
                "xai_area_ratio_image": float(xai_bin.mean()),
                "xai_path": str(xai_path),
                "pseudo_npz_path": str(pseudo_path),
                "pearson_corr": pearson_corr(xai_map, concept_map),
            }

            for metric in metric_names:
                rv = np.asarray(random_metrics[metric], dtype=float)
                rv = rv[np.isfinite(rv)]

                random_mean = float(rv.mean()) if rv.size > 0 else np.nan
                random_std = float(rv.std(ddof=1)) if rv.size > 1 else np.nan

                base[f"observed_{metric}"] = observed[metric]
                base[f"random_{metric}_mean"] = random_mean
                base[f"random_{metric}_std"] = random_std
                base[f"{metric}_enrichment"] = safe_divide(observed[metric], random_mean)
                base[f"{metric}_z"] = safe_z_score(observed[metric], random_mean, random_std)
                base[f"{metric}_empirical_p_ge"] = empirical_p_greater_or_equal(observed[metric], rv)

            chance_records.append(base)

    # End-of-image checkpoint and cleanup
    if (image_idx + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(chance_records).to_csv(checkpoint_path, index=False)
        pd.DataFrame(load_errors).to_csv(errors_checkpoint_path, index=False)
        print(f"Checkpoint saved at image {image_idx + 1}: {checkpoint_path}")

    for var_name in [
        "lesion_map",
        "lesion_bin",
        "xai_map",
        "xai_bin",
        "concept_map",
        "concept_bin_raw",
        "concept_bin",
        "random_region",
        "sample_metrics",
        "random_metrics",
    ]:
        if var_name in locals():
            del locals()[var_name]

    gc.collect()


chance_metrics = pd.DataFrame(chance_records)
chance_random_distributions = pd.DataFrame(random_distribution_records)
chance_errors = pd.DataFrame(load_errors)

chance_metrics_path = CHANCE_METRICS_DIR / f"phase5_{RUN_NAME}_chance_adjusted_xai_concept_metrics.csv"
chance_random_path = CHANCE_METRICS_DIR / f"phase5_{RUN_NAME}_chance_random_distributions.csv"
chance_errors_path = CHANCE_METRICS_DIR / f"phase5_{RUN_NAME}_chance_load_errors.csv"

chance_metrics.to_csv(chance_metrics_path, index=False)
chance_random_distributions.to_csv(chance_random_path, index=False)
chance_errors.to_csv(chance_errors_path, index=False)

print("Chance-adjusted metric rows:", len(chance_metrics))
print("Random distribution rows:", len(chance_random_distributions))
print("Load/error rows:", len(chance_errors))

print("Saved chance metrics:", chance_metrics_path)
print("Saved random distributions:", chance_random_path)
print("Saved errors:", chance_errors_path)

display(chance_metrics.head())
display(chance_errors.head())

## 7. Summary by XAI method and pseudo-concept

This table aggregates the chance-adjusted outputs by XAI method and pseudo-concept.

The most useful columns for interpretation are usually:

- `mean_dice_enrichment`
- `mean_iou_enrichment`
- `mean_sir_enrichment`
- `mean_dice_z`
- `mean_iou_z`
- `mean_sir_z`
- `frac_dice_enrichment_gt_1`
- `frac_sir_enrichment_gt_1`

Interpretation:

- enrichment > 1: observed alignment is higher than the same-area random lesion baseline.
- enrichment =~ 1: observed alignment is similar to chance.
- enrichment < 1: observed alignment is lower than chance.

In [ ]:
if len(chance_metrics) == 0:
    raise ValueError("No chance-adjusted metrics were generated.")

summary_aggs = {
    "stem": "count",
    "concept_area_ratio_image": "mean",
    "concept_area_ratio_lesion": "mean",
    "xai_area_ratio_image": "mean",
    "observed_iou": "mean",
    "random_iou_mean": "mean",
    "iou_enrichment": "mean",
    "iou_z": "mean",
    "iou_empirical_p_ge": "mean",
    "observed_dice": "mean",
    "random_dice_mean": "mean",
    "dice_enrichment": "mean",
    "dice_z": "mean",
    "dice_empirical_p_ge": "mean",
    "observed_sir": "mean",
    "random_sir_mean": "mean",
    "sir_enrichment": "mean",
    "sir_z": "mean",
    "sir_empirical_p_ge": "mean",
    "observed_tixai": "mean",
    "random_tixai_mean": "mean",
    "tixai_enrichment": "mean",
    "tixai_z": "mean",
    "pearson_corr": "mean",
}

chance_summary = (
    chance_metrics
    .groupby(["xai_method", "concept"])
    .agg(summary_aggs)
    .rename(columns={
        "stem": "n",
        "concept_area_ratio_image": "mean_concept_area_ratio_image",
        "concept_area_ratio_lesion": "mean_concept_area_ratio_lesion",
        "xai_area_ratio_image": "mean_xai_area_ratio_image",
        "observed_iou": "mean_observed_iou",
        "random_iou_mean": "mean_random_iou",
        "iou_enrichment": "mean_iou_enrichment",
        "iou_z": "mean_iou_z",
        "iou_empirical_p_ge": "mean_iou_empirical_p_ge",
        "observed_dice": "mean_observed_dice",
        "random_dice_mean": "mean_random_dice",
        "dice_enrichment": "mean_dice_enrichment",
        "dice_z": "mean_dice_z",
        "dice_empirical_p_ge": "mean_dice_empirical_p_ge",
        "observed_sir": "mean_observed_sir",
        "random_sir_mean": "mean_random_sir",
        "sir_enrichment": "mean_sir_enrichment",
        "sir_z": "mean_sir_z",
        "sir_empirical_p_ge": "mean_sir_empirical_p_ge",
        "observed_tixai": "mean_observed_tixai",
        "random_tixai_mean": "mean_random_tixai",
        "tixai_enrichment": "mean_tixai_enrichment",
        "tixai_z": "mean_tixai_z",
        "pearson_corr": "mean_pearson_corr",
    })
    .reset_index()
)

# Fractions of image-level cases beating random baseline.
flags = (
    chance_metrics
    .assign(
        dice_enrichment_gt_1=lambda d: d["dice_enrichment"] > 1,
        iou_enrichment_gt_1=lambda d: d["iou_enrichment"] > 1,
        sir_enrichment_gt_1=lambda d: d["sir_enrichment"] > 1,
        tixai_enrichment_gt_1=lambda d: d["tixai_enrichment"] > 1,
        dice_empirical_p_le_05=lambda d: d["dice_empirical_p_ge"] <= 0.05,
        iou_empirical_p_le_05=lambda d: d["iou_empirical_p_ge"] <= 0.05,
        sir_empirical_p_le_05=lambda d: d["sir_empirical_p_ge"] <= 0.05,
    )
    .groupby(["xai_method", "concept"])
    .agg(
        frac_dice_enrichment_gt_1=("dice_enrichment_gt_1", "mean"),
        frac_iou_enrichment_gt_1=("iou_enrichment_gt_1", "mean"),
        frac_sir_enrichment_gt_1=("sir_enrichment_gt_1", "mean"),
        frac_tixai_enrichment_gt_1=("tixai_enrichment_gt_1", "mean"),
        frac_dice_empirical_p_le_05=("dice_empirical_p_le_05", "mean"),
        frac_iou_empirical_p_le_05=("iou_empirical_p_le_05", "mean"),
        frac_sir_empirical_p_le_05=("sir_empirical_p_le_05", "mean"),
    )
    .reset_index()
)

chance_summary = chance_summary.merge(flags, on=["xai_method", "concept"], how="left")

chance_summary = chance_summary.sort_values(
    ["concept", "mean_sir_enrichment"],
    ascending=[True, False],
)

chance_summary_path = CHANCE_METRICS_DIR / f"phase5_{RUN_NAME}_chance_summary_by_method_concept.csv"
chance_summary.to_csv(chance_summary_path, index=False)

print("Saved chance summary:", chance_summary_path)
display(chance_summary)

## 8. Dataset and lesion-size summaries

These are useful checks for whether chance-adjusted alignment behaves differently for:

- HAM10000 vs ISIC2018
- very small / tiny / small / normal lesions

In [ ]:
chance_dataset_summary = (
    chance_metrics
    .groupby(["dataset", "xai_method", "concept"])
    .agg(
        n=("stem", "count"),
        mean_observed_dice=("observed_dice", "mean"),
        mean_random_dice=("random_dice_mean", "mean"),
        mean_dice_enrichment=("dice_enrichment", "mean"),
        mean_dice_z=("dice_z", "mean"),
        mean_observed_iou=("observed_iou", "mean"),
        mean_random_iou=("random_iou_mean", "mean"),
        mean_iou_enrichment=("iou_enrichment", "mean"),
        mean_iou_z=("iou_z", "mean"),
        mean_observed_sir=("observed_sir", "mean"),
        mean_random_sir=("random_sir_mean", "mean"),
        mean_sir_enrichment=("sir_enrichment", "mean"),
        mean_sir_z=("sir_z", "mean"),
        mean_concept_area_ratio_lesion=("concept_area_ratio_lesion", "mean"),
    )
    .reset_index()
    .sort_values(["dataset", "concept", "mean_sir_enrichment"], ascending=[True, True, False])
)

chance_dataset_summary_path = CHANCE_METRICS_DIR / f"phase5_{RUN_NAME}_chance_summary_by_dataset.csv"
chance_dataset_summary.to_csv(chance_dataset_summary_path, index=False)

print("Saved dataset summary:", chance_dataset_summary_path)
display(chance_dataset_summary.head(30))


if "mask_size_class" in run_resolved.columns:
    chance_with_size = chance_metrics.merge(
        run_resolved[["dataset", "stem", "mask_size_class"]].drop_duplicates(),
        on=["dataset", "stem"],
        how="left",
    )

    chance_size_summary = (
        chance_with_size
        .groupby(["mask_size_class", "xai_method", "concept"])
        .agg(
            n=("stem", "count"),
            mean_observed_dice=("observed_dice", "mean"),
            mean_random_dice=("random_dice_mean", "mean"),
            mean_dice_enrichment=("dice_enrichment", "mean"),
            mean_dice_z=("dice_z", "mean"),
            mean_observed_iou=("observed_iou", "mean"),
            mean_random_iou=("random_iou_mean", "mean"),
            mean_iou_enrichment=("iou_enrichment", "mean"),
            mean_iou_z=("iou_z", "mean"),
            mean_observed_sir=("observed_sir", "mean"),
            mean_random_sir=("random_sir_mean", "mean"),
            mean_sir_enrichment=("sir_enrichment", "mean"),
            mean_sir_z=("sir_z", "mean"),
            mean_concept_area_ratio_lesion=("concept_area_ratio_lesion", "mean"),
        )
        .reset_index()
        .sort_values(["concept", "mask_size_class", "mean_sir_enrichment"], ascending=[True, True, False])
    )

    chance_size_summary_path = CHANCE_METRICS_DIR / f"phase5_{RUN_NAME}_chance_summary_by_size_class.csv"
    chance_size_summary.to_csv(chance_size_summary_path, index=False)

    print("Saved size-class summary:", chance_size_summary_path)
    display(chance_size_summary.head(30))
else:
    print("No mask_size_class column found in run_resolved. Skipping size-class summary.")

## 9. Visual summaries

These plots focus on enrichment. They are often easier to interpret than raw observed values.

Recommended reading:

- A bar above 1 suggests above-chance alignment.
- A bar near 1 suggests chance-level alignment.
- A bar below 1 suggests lower-than-chance alignment.

In [ ]:
plot_metrics = [
    "mean_dice_enrichment",
    "mean_iou_enrichment",
    "mean_sir_enrichment",
    "mean_tixai_enrichment",
]

for metric_name in plot_metrics:
    if metric_name not in chance_summary.columns:
        print(f"Skipping {metric_name}: not found in chance_summary.")
        continue

    pivot = chance_summary.pivot_table(
        index="concept",
        columns="xai_method",
        values=metric_name,
        aggfunc="mean",
    )

    ax = pivot.plot(kind="bar", figsize=(11, 5))
    ax.axhline(1.0, linestyle="--", linewidth=1)
    ax.set_title(f"Chance-adjusted pseudo-concept alignment - {metric_name}")
    ax.set_ylabel(metric_name)
    ax.set_xlabel("Pseudo-concept")
    ax.legend(title="XAI method")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    fig_path = CHANCE_FIG_DIR / f"phase5_{RUN_NAME}_chance_{metric_name}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

## 10. Best and worst chance-adjusted examples

This helps identify image-level cases where an XAI map aligns clearly above or below the same-area random lesion baseline.

In [ ]:
ranking_cols = [
    "dataset",
    "stem",
    "xai_method",
    "concept",

    "observed_dice",
    "random_dice_mean",
    "dice_enrichment",
    "dice_z",
    "dice_empirical_p_ge",

    "observed_iou",
    "random_iou_mean",
    "iou_enrichment",
    "iou_z",
    "iou_empirical_p_ge",

    "observed_sir",
    "random_sir_mean",
    "sir_enrichment",
    "sir_z",
    "sir_empirical_p_ge",

    "concept_area_ratio_lesion",
]

for metric in ["dice_enrichment", "sir_enrichment", "iou_enrichment"]:
    if metric not in chance_metrics.columns:
        continue

    print("\n" + "=" * 80)
    print(f"Top examples by {metric}")
    display(
        chance_metrics[ranking_cols]
        .sort_values(metric, ascending=False)
        .head(15)
    )

    print(f"Bottom examples by {metric}")
    display(
        chance_metrics[ranking_cols]
        .sort_values(metric, ascending=True)
        .head(15)
    )

## 11. Optional label-stratified summary

This is exploratory only.

It should not become a major report section unless the effect is clear and easy to explain.

In [ ]:
if {"label", "label_name"}.issubset(run_resolved.columns):
    label_info = run_resolved[["dataset", "stem", "label", "label_name"]].drop_duplicates()

    chance_label_metrics = chance_metrics.merge(
        label_info,
        on=["dataset", "stem"],
        how="left",
    )

    chance_label_summary = (
        chance_label_metrics
        .groupby(["label_name", "xai_method", "concept"])
        .agg(
            n=("stem", "count"),
            mean_dice_enrichment=("dice_enrichment", "mean"),
            mean_iou_enrichment=("iou_enrichment", "mean"),
            mean_sir_enrichment=("sir_enrichment", "mean"),
            mean_dice_z=("dice_z", "mean"),
            mean_iou_z=("iou_z", "mean"),
            mean_sir_z=("sir_z", "mean"),
            mean_concept_area_ratio_lesion=("concept_area_ratio_lesion", "mean"),
        )
        .reset_index()
        .sort_values(["concept", "label_name", "mean_sir_enrichment"], ascending=[True, True, False])
    )

    chance_label_summary_path = CHANCE_METRICS_DIR / f"phase5_{RUN_NAME}_chance_summary_by_label.csv"
    chance_label_summary.to_csv(chance_label_summary_path, index=False)

    print("Saved label summary:", chance_label_summary_path)
    display(chance_label_summary.head(40))
else:
    print("No label/label_name columns found. Skipping label-stratified summary.")